# Iowa Ev Fiyatlarını Tahmin Etme

## Verileri içe aktar

In [ ]:
import pandas as pd

url = 'https://drive.google.com/uc?export=download&id=1nMrGkRhxr94B0a73gx2dd3HeMe9KCRgq'
df = pd.read_csv(url)
df.head()

In [ ]:
df.shape

## Temizleme - NA değerleri

### Hangi sütunlarda eksik var?

In [ ]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

### %30'dan fazla eksik olan sütunları sil

In [ ]:
threshold = 0.3 * len(df)
df = df.dropna(axis=1, thresh=len(df) - threshold)
df.shape

### Id sütununu sil

In [ ]:
df = df.drop(columns=['Id'])
df.shape

## X ve y'yi seç

In [ ]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

## Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

## Eksik değerleri doldurma

### Sayısal değerler

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='mean').set_output(transform='pandas')

In [ ]:
X_train_num = X_train.select_dtypes(include='number')
X_test_num = X_test.select_dtypes(include='number')

In [ ]:
X_train_num_imputed = num_imputer.fit_transform(X_train_num)
X_test_num_imputed = num_imputer.transform(X_test_num)

In [ ]:
assert X_train_num_imputed.isnull().sum().sum() == 0
assert X_test_num_imputed.isnull().sum().sum() == 0

### Sayısal olmayan değerler

In [ ]:
X_train_cat = X_train.select_dtypes(exclude='number')
X_test_cat = X_test.select_dtypes(exclude='number')

cat_imputer = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
X_train_cat_imputed = cat_imputer.fit_transform(X_train_cat)
X_test_cat_imputed = cat_imputer.transform(X_test_cat)

In [ ]:
assert X_train_cat_imputed.isnull().sum().sum() == 0
assert X_test_cat_imputed.isnull().sum().sum() == 0

## Ölçeklendirme (sayısal)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform='pandas')
X_train_num_scaled = scaler.fit_transform(X_train_num_imputed)
X_test_num_scaled = scaler.transform(X_test_num_imputed)

In [ ]:
assert X_train_num_scaled.isnull().sum().sum() == 0
assert X_test_num_scaled.isnull().sum().sum() == 0

## Kategorik değerleri kodlama

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').set_output(transform='pandas')
X_train_cat_encoded = encoder.fit_transform(X_train_cat_imputed)
X_test_cat_encoded = encoder.transform(X_test_cat_imputed)

In [ ]:
X_train_cat_encoded.head()

## Sayısal ve kategorik verileri birleştir

In [ ]:
X_train_preprocessed = pd.concat(
    [X_train_num_scaled.reset_index(drop=True), X_train_cat_encoded.reset_index(drop=True)],
    axis=1
)
X_test_preprocessed = pd.concat(
    [X_test_num_scaled.reset_index(drop=True), X_test_cat_encoded.reset_index(drop=True)],
    axis=1
)
X_train_preprocessed.shape

## Doğrusal regresyon modeli

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_preprocessed, y_train)

## MAE hesaplama

In [ ]:
from sklearn.metrics import mean_absolute_error

y_pred = model.predict(X_test_preprocessed)
mean_absolute_error(y_test, y_pred)

## Baseline karşılaştırma (ortalama tahmini)

In [ ]:
y_pred_mean = [y_train.mean() for y in y_test]
mean_absolute_error(y_test, y_pred_mean)